In [142]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [143]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

import os

In [144]:
df = pd.read_csv("/content/processed_v2.csv")

df = df.rename(columns={"processed_text": "text"})

print("Original size:", df.shape)
df.head()

Original size: (12759, 2)


,text,label
0,десятки бригад! операція почалась – новий наст...,Fake
1,дуже важкий грип зараз мандрує україною! у діт...,Fake
2,"виcтyп гeнceкa нато nідірвaв мережу: “цiнa, як...",Fake
3,"залишилися лічені дні, почнеться справжня “м'я...",Fake
4,"кремль втретє змінив тактику щодо україни, теп...",Fake


In [145]:
df = df.dropna(subset=["text"])

df["word_len"] = df["text"].apply(lambda x: len(str(x).split()))

df = df[df["word_len"] >= 30]

texts = df["text"].tolist()

print("After filtering:", df.shape)
df.head()

After filtering: (8947, 3)


,text,label,word_len
0,десятки бригад! операція почалась – новий наст...,Fake,256
1,дуже важкий грип зараз мандрує україною! у діт...,Fake,87
2,"виcтyп гeнceкa нато nідірвaв мережу: “цiнa, як...",Fake,149
3,"залишилися лічені дні, почнеться справжня “м'я...",Fake,132
4,"кремль втретє змінив тактику щодо україни, теп...",Fake,999


In [147]:
stopwords = [
    "і","та","на","що","до","з","за","у","в","не","це","як","для","по", "про","від","але","або","чи","також","його","її","вони","ми",
    "нa", "щo", "тa", "нe", "зa", 'цe', "пpo", "дo", "щօ", 'але', 'по', "він","які","року","під","час","із","після","буде","щоб","aлe",
    "йoгo","тaкoж","пo","вжe","чepeз","вoни","щe","щодо", "можна","був","пам", "про", "дօ", 'чac', 'бyдe', 'тaм', 'тaк', 'дyжe', 'зapaз',
    'хто', 'что','это','єр','the', 'ви', 'бо', 'тут', 'тоді', 'ти', 'нато', 'єс', 'путін'
]

tfidf = TfidfVectorizer(
    max_df=0.05,
    min_df=3,
    stop_words=stopwords
)

count = CountVectorizer(
    max_df=0.05,
    min_df=3,
    stop_words=stopwords
)

X_tfidf = tfidf.fit_transform(texts)
X_count = count.fit_transform(texts)

print("TF-IDF shape:", X_tfidf.shape)
print("Count shape:", X_count.shape)

TF-IDF shape: (8947, 31331)
Count shape: (8947, 31331)


In [148]:
def get_top_words(model, feature_names, n_top=10):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_features = [feature_names[i] for i in topic.argsort()[:-n_top - 1:-1]]
        topics.append(top_features)
    return topics


def print_topics(topics):
    for i, words in enumerate(topics):
        print(f"Topic {i+1}: {', '.join(words)}")

In [149]:
lsa_3 = TruncatedSVD(n_components=3, random_state=42)
lsa_5 = TruncatedSVD(n_components=5, random_state=42)

lsa3_topics = lsa_3.fit_transform(X_tfidf)
lsa5_topics = lsa_5.fit_transform(X_tfidf)

lsa3_words = get_top_words(lsa_3, tfidf.get_feature_names_out())
lsa5_words = get_top_words(lsa_5, tfidf.get_feature_names_out())

print("LSA k=3")
print_topics(lsa3_words)

print("\nLSA k=5")
print_topics(lsa5_words)

LSA k=3
Topic 1: внаслідок, удару, атаки, допомоги, війська, тисяч, кількість, міністр, квітня, військової
Topic 2: внаслідок, удару, ова, атаки, постраждалих, загиблих, обстрілу, загинули, районі, ракетного
Topic 3: напрямку, ппо, військ, знищили, ракет, бпла, ворог, війська, ракети, збили

LSA k=5
Topic 1: внаслідок, удару, атаки, допомоги, війська, тисяч, кількість, міністр, квітня, військової
Topic 2: внаслідок, удару, ова, атаки, постраждалих, загиблих, обстрілу, загинули, районі, ракетного
Topic 3: напрямку, військ, ппо, знищили, бпла, ворог, війська, ракет, населених, донецької
Topic 4: допомоги, євро, млрд, млн, пакет, грн, доларів, військової, ппо, міністр
Topic 5: канали, повідомляють, змі, посиланням, допомоги, пишуть, повідомлення, пакет, telegram, джерела


In [150]:
lda_3 = LatentDirichletAllocation(n_components=3, random_state=42)
lda_5 = LatentDirichletAllocation(n_components=5, random_state=42)

lda3_topics = lda_3.fit_transform(X_count)
lda5_topics = lda_5.fit_transform(X_count)

lda3_words = get_top_words(lda_3, count.get_feature_names_out())
lda5_words = get_top_words(lda_5, count.get_feature_names_out())

print("\nLDA k=3")
print_topics(lda3_words)

print("LDA k=5")
print_topics(lda5_words)


LDA k=3
Topic 1: внаслідок, напрямку, атаки, удару, війська, районі, ударів, росіяни, ворог, окупанти
Topic 2: нас, багато, питання, тільки, міністр, польщі, люди, треба, потрібно, просто
Topic 3: грн, році, млн, дітей, допомоги, євро, гривень, тисяч, суд, президента
LDA k=5
Topic 1: тцк, поліції, служби, мобілізації, сбу, зв, квітня, суд, осіб, поліція
Topic 2: нас, багато, треба, просто, питання, тільки, люди, потрібно, нам, році
Topic 3: допомоги, країн, міністр, році, президента, млн, прем, підтримку, безпеки, євро
Topic 4: укpaїни, канали, 16, аес, повідомляють, навчання, змі, пвк, посиланням, pф
Topic 5: напрямку, внаслідок, атаки, війська, удару, росіяни, військ, окупанти, районі, ударів


In [151]:
def get_top_docs(topic_matrix, texts, top_n=3):
    top_docs = {}

    for topic_idx in range(topic_matrix.shape[1]):
        topic_scores = topic_matrix[:, topic_idx]
        top_indices = topic_scores.argsort()[-top_n:][::-1]

        top_docs[topic_idx] = [texts[i] for i in top_indices]

    return top_docs


lsa5_docs = get_top_docs(lsa5_topics, texts)
lda5_docs = get_top_docs(lda5_topics, texts)

In [152]:
for topic, docs in lsa5_docs.items():
    print(f"\nLSA Topic {topic+1}:")
    for d in docs:
        print("-", d[:200])

for topic, docs in lda5_docs.items():
    print(f"\nLDA Topic {topic+1}:")
    for d in docs:
        print("-", d[:200])


LSA Topic 1:
- на цьому і не тільки фокусували увагу світові змі станом на ранок 6 жовтня. сша та європа засудили удар рф у грозі, а кремль заперечує білий дім засудив напад на кафе та продуктовий магазин в українсь
- ми записуємо це інтерв'ю в момент (інтерв'ю було записане 22 грудня, - ред.), коли третя окрема штурмова бригада перебуває не на фронті, тому що ви зараз працюєте, вдосконалюєтеся, доукомплектовуєтеся
- віталій портников: говорити під кінець 2023 року з істориком, напевно, досить непроста задача. якщо ми уважно подивимося на диспозицію того, що відбувалося у 2023 році з точки зору історії, то ми може

LSA Topic 2:
- окупанти вбили 7-річну дитину на сумщині та випустили понад 770 снарядів по херсонщині: як минула ніч ніч проти 29 листопада у більшості областей україни минула відносно спокійно. обласні державні адм
- через нічну атаку рф дронами постраждали шестеро людей на дніпропетровщині та харківщині, повідомляє місцева влада. голова дніпропетровської обласної військов

In [153]:
os.makedirs("docs", exist_ok=True)

audit = """
Lab 8 Audit Summary

Original size: (12759, 2)
After filtering: (8947, 3)

Models:
LSA (k=3, k=5)
LDA (k=3, k=5)

Steps:
фільтрація
TF-IDF / Count vectorization
topic extraction
top words analysis
top documents analysis

Findings:
теми доволі шумні і overlapping
і LSA, і LDA дають майже однаковий результат
LDA має більше беззмістовних тем
LSA наче корисніша, але загалом теми не надто різні, зокрема через занадто однорідний корпус. Майже всі тексти на військову тематику, тож поділ на теми для такої задачі бінарної класифікації на цьому датасеті не надто інформативний.
"""

with open("docs/audit_summary_lab8.md", "w") as f:
    f.write(audit)

In [154]:
notes = """
Topic Notes Lab 8

tfidf = TfidfVectorizer(
    max_df=0.05,
    min_df=3,
    stop_words=stopwords
)

count = CountVectorizer(
    max_df=0.05,
    min_df=3,
    stop_words=stopwords
)

X_tfidf = tfidf.fit_transform(texts)
X_count = count.fit_transform(texts)

print("TF-IDF shape:", X_tfidf.shape)
print("Count shape:", X_count.shape)

LSA

lsa_3 = TruncatedSVD(n_components=3, random_state=42)
lsa_5 = TruncatedSVD(n_components=5, random_state=42)

lsa3_topics = lsa_3.fit_transform(X_tfidf)
lsa5_topics = lsa_5.fit_transform(X_tfidf)

Topic 1: внаслідок, удару, атаки, допомоги, війська, тисяч, кількість, міністр, квітня, військової
Name: Військові втрати та допомога
Description: Тема стосується наслідків атак і втрат, а також міжнародної допомоги. Слова «внаслідок», «удару», «атаки» вказують на бойові дії, а «допомоги», «міністр», «війська» — на реакцію та підтримку. Це поєднання військових подій і політичних рішень.

Topic 2: внаслідок, удару, ова, атаки, постраждалих, загиблих, обстрілу, загинули, районі, ракетного
Name: Жертви та обстріли
Description: Тут акцент на людських втрат і наслідках атак. Слова «постраждалих», «загиблих», «загинули», «районі» показують фокус на жертвах, а «ракетного», «обстрілу» — на характері атак. Назва відображає головну ідею — втрати серед населення.

Topic 3: напрямку, військ, ппо, знищили, бпла, ворог, війська, ракет, населених, донецької
Name: Військові операції та ППО
Description: Тема зосереджена на військових діях і протиповітряній обороні. Слова «напрямку», «військ», «ппо», «знищили», «бпла» вказують на активні бойові операції та використання оборонних систем. Це про тактику і захист.

Topic 4: допомоги, євро, млрд, млн, пакет, грн, доларів, військової, ппо, міністр
Name: Фінансова допомога та озброєння
Description: Тут йдеться про фінансову підтримку та військові пакети. Слова «євро», «млрд», «млн», «грн», «доларів» показують економічний аспект, а «пакет», «військової», «ппо» — про військову складову. Назва відображає поєднання грошей і оборони.

Topic 5: канали, повідомляють, змі, посиланням, допомоги, пишуть, повідомлення, пакет, telegram, джерела
Name: Інформаційні канали та повідомлення
Description: Тема стосується поширення інформації через медіа. Слова «канали», «повідомляють», «змі», «telegram», «джерела» вказують на джерела новин, а «допомоги», «пакет» — на зміст повідомлень. Це про інформаційний простір і комунікацію.

LDA

lda_3 = LatentDirichletAllocation(n_components=3, random_state=42)
lda_5 = LatentDirichletAllocation(n_components=5, random_state=42)

lda3_topics = lda_3.fit_transform(X_count)
lda5_topics = lda_5.fit_transform(X_count)

Topic 1: тцк, поліції, служби, мобілізації, сбу, зв, квітня, суд, осіб, поліція
Name: Мобілізація та правоохоронні органи
Description: Тема охоплює мобілізацію, суди та діяльність силових структур. Слова «тцк», «поліції», «служби», «мобілізації», «сбу» підтверджують зв’язок із правоохоронними та військовими органами. Назва відображає юридичний і силовий контекст.

Topic 3: допомоги, країн, міністр, році, президента, млн, прем, підтримку, безпеки, євро
Name: Міжнародна допомога та політика
Description: Тема стосується міжнародної підтримки та політичних рішень. Слова «допомоги», «країн», «міністр», «президента», «євро», «млн» підтверджують акцент на фінансовій і політичній допомозі. Назва відображає міжнародний вимір.

Topic 5: напрямку, внаслідок, атаки, війська, удару, росіяни, військ, окупанти, районі, ударів
Name: Військові атаки та окупанти
Description: Тема зосереджена на бойових діях і ворогові. Слова «напрямку», «внаслідок», «атаки», «війська», «удару», «росіяни», «окупанти» підтверджують військовий контекст. Назва відображає агресію та наслідки атак.


Problematic topics


Topic 2: нас, багато, треба, просто, питання, тільки, люди, потрібно, нам, році
Name: -
Description: тема надто загальна і не дає реального змісту

Topic 4: укpaїни, канали, 16, аес, повідомляють, навчання, змі, пвк, посиланням, pф
Name: -
Description: змішано різні сюжети, відсутній зміст

Summary
теми доволі шумні і overlapping
і LSA, і LDA дають майже однаковий результат
LDA має більше беззмістовних тем
LSA наче корисніша, але загалом теми не надто різні, зокрема через занадто однорідний корпус. Майже всі тексти на військову тематику, тож поділ на теми для такої задачі бінарної класифікації на цьому датасеті не надто інформативний.
"""

with open("docs/topic_notes_lab8.md", "w") as f:
    f.write(notes)